# Analyze one exocytosis TIFF stack

Run the cells from top to bottom in VS Code with a Python Jupyter kernel. This notebook calls the Python pipeline in this folder to generate a cell mask (if needed), preprocess one TIFF stack, detect spots, track events, extract fluorescence traces, and summarize the results.

Before running, install dependencies from `python_pipeline/requirements.txt` in the selected kernel. Edit the TIFF path in the next cell. If you already have a mask, set `MASK_PATH` to its path; otherwise leave it as `None` and the notebook will generate one.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from IPython.display import display

# EDIT THIS: full path to one grayscale, multipage TIFF stack.
TIFF_PATH = Path(r"C:\path\to\your_movie.tif")

# Optional: point to an existing binary cell-mask TIFF. Keep None to generate one.
MASK_PATH = None  # Example: Path(r"C:\data\MaskFiles\your_movie_mask_file.tif")

# Locate the repository root so imports work whether VS Code starts in the repo
# root or in the python_pipeline folder.
if Path.cwd().name == "python_pipeline":
    REPOSITORY_ROOT = Path.cwd().parent
elif (Path.cwd() / "python_pipeline").is_dir():
    REPOSITORY_ROOT = Path.cwd()
else:
    raise FileNotFoundError(
        "Open this notebook from the repository root or python_pipeline folder. "
        "Alternatively, add the repository root to the Python path."
    )

if not TIFF_PATH.is_file():
    raise FileNotFoundError(f"Set TIFF_PATH to an existing TIFF stack: {TIFF_PATH}")

if MASK_PATH is not None:
    MASK_PATH = Path(MASK_PATH).expanduser().resolve()

sys.path.insert(0, str(REPOSITORY_ROOT))
from python_pipeline.image_processing import create_cell_mask, iter_tiff_frames, read_binary_mask
from python_pipeline.pipeline import (
    detect_movie,
    extract_fluorescence_traces,
    preprocess_movie,
)
from python_pipeline.tracking import track_binary_tiff

TIFF_PATH = TIFF_PATH.expanduser().resolve()
ANALYSIS_DIR = TIFF_PATH.parent / f"{TIFF_PATH.stem}_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

if MASK_PATH is None:
    conventional_mask = TIFF_PATH.parent.parent / "MaskFiles" / f"{TIFF_PATH.stem}_mask_file.tif"
    MASK_PATH = conventional_mask if conventional_mask.is_file() else ANALYSIS_DIR / f"{TIFF_PATH.stem}_mask_file.tif"

print(f"Input TIFF: {TIFF_PATH}")
print(f"Cell mask:  {MASK_PATH}")
print(f"Outputs:    {ANALYSIS_DIR}")

## 1. Check the movie and prepare its cell mask

The automatic mask generator follows the MATLAB approach: compare the first frame with the stack average, smooth and sharpen the difference, threshold it, and retain the largest connected component. Inspect the generated or supplied mask before proceeding.

In [ ]:
with tifffile.TiffFile(TIFF_PATH) as tif:
    frame_count = len(tif.pages)
    frame_shape = tif.pages[0].shape if frame_count else None

if frame_count < 10:
    raise ValueError(f"Detection requires at least 10 frames; this TIFF has {frame_count}.")
if frame_shape is None or len(frame_shape) != 2:
    raise ValueError(f"Expected grayscale 2D TIFF frames; found page shape {frame_shape}.")

if not MASK_PATH.is_file():
    print("No mask found; generating a mask from this TIFF stack.")
    create_cell_mask(TIFF_PATH, MASK_PATH)

cell_mask = read_binary_mask(MASK_PATH)
if cell_mask.shape != frame_shape:
    raise ValueError(f"Mask shape {cell_mask.shape} does not match TIFF frames {frame_shape}.")

first_frame = next(iter_tiff_frames(TIFF_PATH))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(first_frame, cmap="gray")
axes[0].set_title("Raw TIFF — frame 1")
axes[1].imshow(cell_mask, cmap="gray")
axes[1].set_title(f"Cell mask ({cell_mask.sum():,} pixels)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Preprocess, detect, and track events

Preprocessing subtracts the median outside-mask background, scales each frame to 8-bit, and histogram-matches later frames to the first. Detection starts at frame 10, subtracts the median of the five preceding frames, computes a multiscale difference-of-Gaussians response, and thresholds using the stack-level median of per-frame maxima. Connected components of at least 25 pixels are linked with constant-velocity Kalman prediction and global distance assignment.

In [ ]:
preprocessed_path = ANALYSIS_DIR / f"{TIFF_PATH.stem}_PreProcessed.tif"
response_path = ANALYSIS_DIR / f"{TIFF_PATH.stem}_DetectionResponse.tif"
binary_path = ANALYSIS_DIR / f"{TIFF_PATH.stem}_Processed.tif"
tracking_path = ANALYSIS_DIR / f"{TIFF_PATH.stem}_Processed_tracking.csv"
traces_path = ANALYSIS_DIR / f"{TIFF_PATH.stem}_fluorescence_traces.csv"

preprocess_movie(TIFF_PATH, MASK_PATH, preprocessed_path)
threshold = detect_movie(preprocessed_path, MASK_PATH, response_path, binary_path)
tracks = track_binary_tiff(binary_path, tracking_path, minimum_blob_area=25)
traces = extract_fluorescence_traces(preprocessed_path, tracking_path, traces_path)

print(f"Frames analyzed: {frame_count - 9} (starting at raw frame 10)")
print(f"Detection threshold: {threshold:.4g}")
print(f"Tracked events: {len(tracks)}")
print(f"Saved outputs under: {ANALYSIS_DIR}")

## 3. Review event measurements

Track `time` is one-based relative to the detection TIFF, whose first page corresponds to raw frame 10. The trace CSV's `frame_num` maps the event's first detection back to the raw TIFF frame. Each trace contains 20 frames before through 30 frames after that event frame, as a 21×21-pixel ROI mean. The trace time axis below is in frames; convert to seconds only if the acquisition frame interval is known.

In [ ]:
if tracks.empty:
    print("No connected components passed the detection and minimum-area filters.")
else:
    summary_columns = [
        "id", "centroid_1", "centroid_2", "time", "lifetime",
        "visible_frames", "area", "major_axis", "minor_axis", "perimeter",
    ]
    display(tracks[summary_columns].head(20))
    display(tracks[["lifetime", "visible_frames", "area"]].describe().round(2))

    trace_columns = [column for column in traces.columns if column.startswith("TimePoint_")]
    relative_frames = np.arange(-20, 31)
    fig, ax = plt.subplots(figsize=(10, 5))
    for _, event in traces.head(20).iterrows():
        ax.plot(relative_frames, event[trace_columns].to_numpy(dtype=float), alpha=0.65, linewidth=1)
    ax.axvline(0, color="black", linestyle="--", linewidth=1, label="First detection")
    ax.set(xlabel="Frames relative to first detection", ylabel="Mean ROI intensity (8-bit)",
           title="Fluorescence traces (first 20 events)")
    ax.legend()
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

## 4. Inspect the detection and event locations

The left panel shows the binary detection page for the first tracked event. The right panel shows that event's ROI center on the corresponding preprocessed raw frame. Coordinates are plotted as pixel columns (`x`) and rows (`y`).

In [ ]:
if tracks.empty:
    print("There are no tracks to overlay.")
else:
    event = tracks.iloc[0]
    detection_page = int(event["time"])
    raw_page = detection_page + 9
    with tifffile.TiffFile(binary_path) as tif:
        detection_image = tif.pages[detection_page - 1].asarray()
    with tifffile.TiffFile(preprocessed_path) as tif:
        raw_processed_image = tif.pages[raw_page - 1].asarray()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(detection_image, cmap="gray")
    axes[0].set_title(f"Binary detections — detection page {detection_page}")
    axes[1].imshow(raw_processed_image, cmap="gray")
    axes[1].scatter(event["centroid_1"], event["centroid_2"], s=100,
                    facecolors="none", edgecolors="red", linewidths=2)
    axes[1].set_title(f"Preprocessed raw frame {raw_page} — event {int(event['id'])}")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Output files

The notebook writes results to a separate `<TIFF-name>_analysis` folder next to the input TIFF. It contains the mask (if generated), preprocessed stack, multiscale detection responses, binary detections, tracking CSV, and fluorescence-trace CSV. This keeps the one-movie analysis separate from batch-run output folders.

In [ ]:
for path in sorted(ANALYSIS_DIR.iterdir()):
    print(path.name)